# 📷 ➜ ▶️ Image to Sketch Animation — Colab Runner

Run this notebook on a **GPU runtime (T4 or better)** for best performance.

> **Before you start:** Go to `Runtime` → `Change runtime type` → select **T4 GPU**.

---
### Workflow
1. Run **Cell 1** once per session → verifies GPU, clones repo, installs deps, patches app.
2. Run **Cell 2** each time you want to process an image → full interactive UI.

---
## Cell 1 — Setup (Run once per session)
▶️ Run this cell **once** at the start of every session.

In [ ]:
import os, re

# ── 1. GPU STATUS ──────────────────────────────────────────
print("=" * 55)
print("1/4  GPU STATUS")
print("=" * 55)
!nvidia-smi

# ── 2. CLONE REPO ──────────────────────────────────────────
print("\n" + "=" * 55)
print("2/4  REPOSITORY SETUP")
print("=" * 55)
REPO_URL = "https://github.com/daslearning-org/image-to-animation-offline.git"
REPO_DIR = "/content/image-to-animation-offline"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
    print("✅ Repository cloned.")
else:
    print("✅ Repository already exists, skipping clone.")

# ── 3. INSTALL DEPENDENCIES ────────────────────────────────
print("\n" + "=" * 55)
print("3/4  INSTALLING DEPENDENCIES")
print("=" * 55)
!pip install -q numpy opencv-python av ipywidgets
print("✅ Dependencies ready.")

# ── 4. PATCH sketchApi.py ──────────────────────────────────
# This automatically fixes sketchApi.py whether it is the old GitHub
# version (Kivy imported directly, no GPU support) or the updated
# local version (already has our changes). Patches are skipped safely
# if the feature is already present.
print("\n" + "=" * 55)
print("4/4  PATCHING sketchApi.py")
print("=" * 55)

SKETCH_PATH = f"{REPO_DIR}/kivy/sketchApi.py"

with open(SKETCH_PATH, 'r') as f:
    content = f.read()

patched = False

# ── Patch 1: Wrap Kivy import in try/except ─────────────────
if 'KIVY_AVAILABLE' not in content:
    content = content.replace(
        'from kivy.clock import Clock',
        '# Optional Kivy import (only needed for GUI mode)\n'
        'try:\n'
        '    from kivy.clock import Clock\n'
        '    KIVY_AVAILABLE = True\n'
        'except ImportError:\n'
        '    Clock = None\n'
        '    KIVY_AVAILABLE = False'
    )
    print("🔧 Patch 1 applied: Kivy import wrapped in try/except.")
    patched = True
else:
    print("✅ Patch 1 skipped: Kivy try/except already present.")

# ── Patch 2: Add CuPy GPU block ─────────────────────────────
if '_GPU_AVAILABLE' not in content:
    CUPY_BLOCK = (
        '\n'
        '# Optional CuPy import for GPU acceleration (Nvidia CUDA).\n'
        '# If CuPy is not installed or no CUDA device is found,\n'
        '# the app falls back to CPU (NumPy) silently with no errors.\n'
        'try:\n'
        '    import cupy as cp\n'
        '    _GPU_AVAILABLE = cp.cuda.runtime.getDeviceCount() > 0\n'
        '    if _GPU_AVAILABLE:\n'
        '        _name_raw = cp.cuda.runtime.getDeviceProperties(0)[\'name\']\n'
        '        _gpu_name = _name_raw.decode() if isinstance(_name_raw, bytes) else _name_raw\n'
        '        print(f"[sketchApi] GPU mode active: {_gpu_name}")\n'
        '    else:\n'
        '        cp = None\n'
        '        _GPU_AVAILABLE = False\n'
        '        print("[sketchApi] CuPy found but no CUDA device. Using CPU (NumPy).")\n'
        'except Exception:\n'
        '    cp = None\n'
        '    _GPU_AVAILABLE = False\n'
        '    print("[sketchApi] CuPy not available. Using CPU (NumPy).")\n'
    )
    # Insert the CuPy block right after the Kivy block ends
    content = content.replace(
        'KIVY_AVAILABLE = False\n\n',
        'KIVY_AVAILABLE = False\n' + CUPY_BLOCK + '\n'
    )
    print("🔧 Patch 2 applied: CuPy GPU block added.")
    patched = True
else:
    print("✅ Patch 2 skipped: CuPy GPU block already present.")

# ── Patch 3: GPU-aware euc_dist ─────────────────────────────
if 'cp.asarray' not in content:
    GPU_EUCDIST = (
        'def euc_dist(arr1, point):\n'
        '    """\n'
        '    Computes Euclidean distance from every row in arr1 to point.\n'
        '    Uses GPU (CuPy) when available, otherwise falls back to CPU (NumPy).\n'
        '    Always returns a plain NumPy array so all callers work unchanged.\n'
        '    """\n'
        '    if _GPU_AVAILABLE:\n'
        '        gpu_arr1 = cp.asarray(arr1)\n'
        '        gpu_point = cp.asarray(point)\n'
        '        square_sub = (gpu_arr1 - gpu_point) ** 2\n'
        '        result = cp.sqrt(cp.sum(square_sub, axis=1))\n'
        '        return cp.asnumpy(result)  # Always return plain NumPy array\n'
        '    else:\n'
        '        # Original CPU logic — untouched\n'
        '        square_sub = (arr1 - point) ** 2\n'
        '        return np.sqrt(np.sum(square_sub, axis=1))'
    )
    content = re.sub(
        r'def euc_dist\(arr1, point\):.*?return np\.sqrt\(np\.sum\(square_sub, axis=1\)\)',
        GPU_EUCDIST,
        content,
        flags=re.DOTALL
    )
    print("🔧 Patch 3 applied: GPU-aware euc_dist injected.")
    patched = True
else:
    print("✅ Patch 3 skipped: GPU-aware euc_dist already present.")

# ── Save if any patch was applied ───────────────────────────
if patched:
    with open(SKETCH_PATH, 'w') as f:
        f.write(content)
    print("\n💾 Patched sketchApi.py saved.")
else:
    print("\n✅ sketchApi.py already up to date, no changes needed.")

# ── Verify CuPy GPU directly ────────────────────────────────
print("\n" + "=" * 55)
print("GPU BACKEND CHECK")
print("=" * 55)
try:
    import cupy as cp
    n = cp.cuda.runtime.getDeviceCount()
    if n > 0:
        name_raw = cp.cuda.runtime.getDeviceProperties(0)['name']
        gpu_name = name_raw.decode() if isinstance(name_raw, bytes) else name_raw
        print(f"✅ GPU ready: {gpu_name}")
        print("   Processing will use GPU acceleration (CuPy).")
    else:
        print("⚠️  CuPy installed but no GPU found. Will run on CPU.")
except Exception as e:
    print(f"⚠️  CuPy not available. Will run on CPU (NumPy). [{e}]")

print("\n✅ Setup complete! Proceed to Cell 2.")

---
## Cell 2 — 🎛️ Interactive Processing UI
▶️ Run this cell each time you want to process an image.

It will show:
- 📤 A **file upload button** to select your image
- ⚙️ **Visual controls** for all settings
- 🚀 A **Run** button that builds and executes the CLI command

In [ ]:
import os, subprocess, glob, sys
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

REPO_DIR   = "/content/image-to-animation-offline"
OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ──────────────────────────────────────────────────────────
# STYLES
# ──────────────────────────────────────────────────────────
display(HTML("""
<style>
  .section-title {
    font-size: 15px; font-weight: bold;
    color: #444; margin: 14px 0 6px 0;
    border-bottom: 2px solid #ddd; padding-bottom: 4px;
  }
  .hint { font-size: 12px; color: #888; margin-top: 2px; }
</style>
"""))

# ──────────────────────────────────────────────────────────
# STEP 1 — FILE UPLOAD
# ──────────────────────────────────────────────────────────
display(HTML('<div class="section-title">📤 Step 1 — Select Image</div>'))

upload_btn = widgets.FileUpload(
    accept='.jpg,.jpeg,.png,.webp',
    multiple=False,
    description='Choose Image',
    button_style='info',
    layout=widgets.Layout(width='220px')
)
upload_status = widgets.Label(value='⚠️  No image selected yet.')
display(widgets.HBox([upload_btn, upload_status]))

UPLOADED_PATH = [None]  # mutable container shared across callbacks

def on_upload_change(change):
    if upload_btn.value:
        fname, fdata = next(iter(upload_btn.value.items()))
        save_path = f"/content/{fname}"
        with open(save_path, 'wb') as f:
            f.write(fdata['content'])
        UPLOADED_PATH[0] = save_path
        upload_status.value = f'✅  Saved: {fname}'

upload_btn.observe(on_upload_change, names='value')

# ──────────────────────────────────────────────────────────
# STEP 2 — VIDEO SETTINGS
# ──────────────────────────────────────────────────────────
display(HTML('<div class="section-title">🎬 Step 2 — Video Settings</div>'))

w_frame_rate = widgets.IntSlider(
    value=24, min=12, max=60, step=1,
    description='Frame Rate:', style={'description_width': '130px'},
    layout=widgets.Layout(width='450px')
)
w_duration = widgets.IntSlider(
    value=1, min=1, max=10, step=1,
    description='End Duration (s):', style={'description_width': '130px'},
    layout=widgets.Layout(width='450px')
)
display(w_frame_rate)
display(HTML('<div class="hint">Frames per second of the output video.</div>'))
display(w_duration)
display(HTML('<div class="hint">How many seconds the final image is held at the end.</div>'))

# ──────────────────────────────────────────────────────────
# STEP 3 — DRAWING SPEED
# ──────────────────────────────────────────────────────────
display(HTML('<div class="section-title">✏️ Step 3 — Drawing Speed</div>'))

w_obj_skip = widgets.IntSlider(
    value=2, min=1, max=10, step=1,
    description='Object Skip:', style={'description_width': '130px'},
    layout=widgets.Layout(width='450px')
)
w_bck_skip = widgets.IntSlider(
    value=3, min=1, max=15, step=1,
    description='Background Skip:', style={'description_width': '130px'},
    layout=widgets.Layout(width='450px')
)
w_fill_speed = widgets.IntSlider(
    value=5, min=1, max=15, step=1,
    description='Fill Speed:', style={'description_width': '130px'},
    layout=widgets.Layout(width='450px')
)
display(w_obj_skip)
display(HTML('<div class="hint">Lower = more detail but slower. Higher = faster but fewer frames captured.</div>'))
display(w_bck_skip)
display(HTML('<div class="hint">Same as above but for the background region.</div>'))
display(w_fill_speed)
display(HTML('<div class="hint">Speed multiplier for the color fill pass.</div>'))

# ──────────────────────────────────────────────────────────
# STEP 4 — SPLIT LENGTH
# ──────────────────────────────────────────────────────────
display(HTML('<div class="section-title">📐 Step 4 — Split Length (Drawing Block Size)</div>'))

w_split_mode = widgets.ToggleButtons(
    options=['Auto (Recommended)', 'Range', 'Exact Value'],
    value='Auto (Recommended)',
    description='Mode:',
    style={'description_width': '80px', 'button_width': '160px'}
)
w_split_range_min = widgets.IntText(value=10, description='Min:', layout=widgets.Layout(width='120px'))
w_split_range_max = widgets.IntText(value=40, description='Max:', layout=widgets.Layout(width='120px'))
w_split_exact = widgets.IntText(value=20, description='Value:', layout=widgets.Layout(width='140px'))

split_range_box = widgets.HBox([w_split_range_min, w_split_range_max])
split_exact_box = widgets.HBox([w_split_exact])
split_range_box.layout.display = 'none'
split_exact_box.layout.display = 'none'

def on_split_mode_change(change):
    split_range_box.layout.display = 'flex' if change['new'] == 'Range' else 'none'
    split_exact_box.layout.display = 'flex' if change['new'] == 'Exact Value' else 'none'

w_split_mode.observe(on_split_mode_change, names='value')
display(w_split_mode)
display(split_range_box)
display(split_exact_box)
display(HTML('<div class="hint">Controls the size of each drawing block. Larger = coarser/faster. Smaller = finer/slower.</div>'))

# ──────────────────────────────────────────────────────────
# STEP 5 — DRAWING MODE
# ──────────────────────────────────────────────────────────
display(HTML('<div class="section-title">🎨 Step 5 — Drawing Mode</div>'))

w_draw_color = widgets.Checkbox(value=False, description='Draw in Color  (use original colors instead of grayscale)')
w_two_pass   = widgets.Checkbox(value=False, description='Two-Pass  (draw edges first, then fill color separately)')
w_end_color  = widgets.Checkbox(value=True,  description='Show Color at End  (reveal full color image at the final frame)')
w_h264       = widgets.Checkbox(value=True,  description='Convert to H264  (better compatibility with players & phones)')

display(w_draw_color)
display(w_two_pass)
display(w_end_color)
display(w_h264)

# ──────────────────────────────────────────────────────────
# STEP 6 — OUTPUT FOLDER
# ──────────────────────────────────────────────────────────
display(HTML('<div class="section-title">💾 Step 6 — Output</div>'))

w_output_dir = widgets.Text(
    value='/content/output',
    description='Output Folder:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='450px')
)
display(w_output_dir)

# ──────────────────────────────────────────────────────────
# STEP 7 — RUN
# ──────────────────────────────────────────────────────────
display(HTML('<div class="section-title">🚀 Step 7 — Run</div>'))

run_btn = widgets.Button(
    description='▶  Start Processing',
    button_style='success',
    layout=widgets.Layout(width='220px', height='40px')
)
cmd_preview = widgets.Textarea(
    value='',
    description='CLI Command:',
    disabled=True,
    style={'description_width': '100px'},
    layout=widgets.Layout(width='680px', height='60px')
)
run_output = widgets.Output()

display(run_btn)
display(cmd_preview)
display(run_output)

def on_run_clicked(b):
    with run_output:
        clear_output(wait=True)

        if UPLOADED_PATH[0] is None:
            print("❌ No image selected. Please upload an image in Step 1 first.")
            return

        out_dir = w_output_dir.value.strip() or '/content/output'
        os.makedirs(out_dir, exist_ok=True)

        cmd = [
            sys.executable,
            f'{REPO_DIR}/kivy/cli.py',
            '-i', UPLOADED_PATH[0],
            '-o', out_dir,
            '-fr', str(w_frame_rate.value),
            '-os', str(w_obj_skip.value),
            '-bs', str(w_bck_skip.value),
            '-d',  str(w_duration.value),
            '-fs', str(w_fill_speed.value),
        ]

        if w_split_mode.value == 'Range':
            cmd += ['-slr', f'{w_split_range_min.value}-{w_split_range_max.value}']
        elif w_split_mode.value == 'Exact Value':
            cmd += ['-sl', str(w_split_exact.value)]

        if w_draw_color.value: cmd.append('-dc')
        if w_two_pass.value:   cmd.append('-tp')
        if w_h264.value:       cmd.append('-h264')

        cmd_preview.value = ' '.join(cmd)

        print('=' * 55)
        print('🚀 Starting processing...')
        print('=' * 55)

        result = subprocess.run(cmd, text=True)

        print('=' * 55)
        if result.returncode == 0:
            videos = glob.glob(f"{out_dir}/*.mp4")
            print(f'\n✅ Done! {len(videos)} video(s) saved to: {out_dir}')
            for v in videos:
                print(f'   📹 {os.path.basename(v)}')
            print('\n📥 Downloading...')
            from google.colab import files
            for v in videos:
                files.download(v)
        else:
            print(f'\n❌ Processing failed. Return code: {result.returncode}')

run_btn.on_click(on_run_clicked)

---
## Cell 3 (Optional) — Save to Google Drive
Run this cell to copy all generated videos to your Google Drive.

In [ ]:
from google.colab import drive
import shutil, glob, os

drive.mount('/content/drive')

DRIVE_DEST = "/content/drive/MyDrive/ImageToAnimation_Output"
os.makedirs(DRIVE_DEST, exist_ok=True)

OUTPUT_DIR = "/content/output"
videos = glob.glob(f"{OUTPUT_DIR}/*.mp4")

if not videos:
    print("⚠️  No videos found. Run Cell 2 first.")
else:
    for v in videos:
        dest = os.path.join(DRIVE_DEST, os.path.basename(v))
        shutil.copy2(v, dest)
        print(f"✅ Saved to Drive: {dest}")

---
## Cell 4 (Optional) — GPU Memory Monitor

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.free,memory.total,utilization.gpu --format=csv